# T Industries - synthetic source data (bronze)
Seeded (reproducible) generator for the T Industries demo. Lands **simulated source-system data** as `bronze_*` tables in `<catalog>.t_hardware` and raw files in the `<catalog>.t_hardware.landing` volume:

* **ERP**: `bronze_products` (285 SKUs), `bronze_customers` (500), `bronze_sales_orders` + `bronze_order_lines` (24 months, seasonality), `bronze_inventory` (Delta BC, Toronto)
* **Price scraping**: `bronze_competitor_prices` (NorthBuild Supply, Pacific Hardware Co, DoorPro Direct; 1-2 equivalents per SKU, monthly)
* **HubSpot (Lakeflow Connect style)**: `bronze_hubspot_companies`, `bronze_hubspot_deals`, `bronze_hubspot_tickets`, `bronze_hubspot_attachments` (+ files in `hubspot_attachments/`)
* **SharePoint (Lakeflow Connect style)**: PDFs/DOCX/markdown in `sharepoint/` mirrored into `bronze_sharepoint_files` (file_name, path, modified_time, content BINARY)
* **Product images** in `product_images/` (64 SKUs). `photo_uploads/` holds hand-curated customer photos (other brands) and is not generated here

Bronze is intentionally a bit messy (SKU formatting, region spellings, mixed date formats, ~1% duplicate lines) so the silver layer of pipeline `t_genie_ready` has real cleaning to do.

Planted stories: exit devices priced >10% above NorthBuild (and losing ON share since Apr 2026); Toronto door-closer backorders since Aug 2026; Matte Black faucet finish-defect tickets and returns; spring/summer construction seasonality.

Idempotent: skips if bronze already exists unless `regenerate = true`.

In [0]:
%pip install -q faker==33.1.0 reportlab==4.2.5 python-docx==1.1.2

In [0]:
dbutils.widgets.text("catalog", "mdk_demo_catalog")
dbutils.widgets.dropdown("regenerate", "false", ["false", "true"])
C = dbutils.widgets.get("catalog")
REGEN = dbutils.widgets.get("regenerate") == "true"
B = f"{C}.t_hardware"                      # one schema; layers are table-name prefixes (bronze_, silver_, gold_, gov_)
VOL = f"/Volumes/{C}/t_hardware/landing"

spark.sql(f"""CREATE SCHEMA IF NOT EXISTS {B} COMMENT 'T Industries demo (TAI backend). One schema, layers by prefix: bronze_ (raw sources), silver_ (cleaned, pipeline t_genie_ready), gold_ (Genie/BI-ready), gov_ (row-level security); metric view t_sales_metrics; volume landing (raw files).'""")
spark.sql(f"""CREATE VOLUME IF NOT EXISTS {B}.landing COMMENT 'Raw files: product_images/, sharepoint/ (simulated SharePoint library), hubspot_attachments/ (simulated HubSpot attachments), photo_uploads/ (photos users ask TAI about)'""")

# `knowledge` = the SharePoint library as raw files, read directly by the T Docs & SOPs Genie agent (Knowledge Assistant replacement).
# In production a Lakeflow Connect SharePoint connector lands files here; in the demo we mirror landing/sharepoint on every run.
spark.sql(f"""CREATE VOLUME IF NOT EXISTS {B}.knowledge COMMENT 'Raw SharePoint documents (spec sheets, installation SOPs, warranty and returns policy, HR handbook excerpt, pricing authority matrix, business rules) read directly by the T Docs & SOPs Genie agent'""")
import os, shutil
def sync_knowledge():
    src, dst = f"{VOL}/sharepoint", f"/Volumes/{C}/t_hardware/knowledge/sharepoint"
    if not os.path.exists(src): return 0
    n = 0
    for d, _, fs in os.walk(src):
        for f in fs:
            s = os.path.join(d, f); t = os.path.join(dst, os.path.relpath(s, src))
            if not os.path.exists(t) or os.path.getsize(t) != os.path.getsize(s) or os.path.getmtime(t) < os.path.getmtime(s):
                os.makedirs(os.path.dirname(t), exist_ok=True); shutil.copyfile(s, t); n += 1
    print(f"knowledge volume: {n} file(s) copied or updated"); return n

if spark.catalog.tableExists(f"{B}.bronze_sharepoint_files") and not REGEN:
    sync_knowledge()
    dbutils.notebook.exit("bronze already exists - skipped (set regenerate=true to rebuild); knowledge volume synced")

In [0]:
import numpy as np, pandas as pd, random, datetime as dt, os, shutil, tempfile
from faker import Faker
SEED = 42
rng = np.random.default_rng(SEED); random.seed(SEED); fake = Faker("en_CA"); Faker.seed(SEED)
TODAY = dt.date(2026, 11, 20)            # fixed demo "today" keeps the story reproducible
START = TODAY - dt.timedelta(days=730)   # 24 months of history

FINISHES = [("US26D","Satin Chrome"),("US32D","Satin Stainless"),("BLK","Matte Black"),("US4","Satin Brass"),("US10B","Oil Rubbed Bronze"),("US26","Polished Chrome")]
LINES = ["Granville","Cambie","Kitsilano","Yaletown","Lonsdale","Kerrisdale","Gastown","Davie","Rideau","Yorkville","Leaside","Annex"]
# category, code, [(subtype, base_price, material, grade_options, n_skus)]
CATS = [
 ("Locks","LK",[("Grade 1 Mortise Lock",420,"Steel / brass",[1],10),("Cylindrical Lever Lock",210,"Steel / zinc",[1,2],10),("Deadbolt",95,"Brass",[1,2,3],10),("Electronic Keypad Lock",540,"Zinc / electronics",[1,2],8)]),
 ("Levers","LV",[("Passage Lever Set",88,"Zinc",[2,3],12),("Privacy Lever Set",96,"Zinc",[2,3],12),("Storeroom Lever Set",180,"Stainless steel",[1,2],10),("Commercial Entry Lever",235,"Stainless steel",[1],8)]),
 ("Door Closers","DC",[("Surface Door Closer",265,"Aluminum",[1],12),("Heavy Duty Parallel Arm Closer",345,"Aluminum / steel",[1],10),("Concealed Overhead Closer",610,"Steel",[1],6),("Residential Door Closer",120,"Aluminum",[3],8)]),
 ("Hinges","HG",[("4.5in Ball Bearing Butt Hinge",28,"Stainless steel",[1,2],14),("Continuous Geared Hinge",310,"Aluminum",[1],8),("Spring Hinge",42,"Steel",[2,3],10),("Pivot Set",260,"Steel",[1],6)]),
 ("Exit Devices","EX",[("Rim Exit Device",640,"Stainless steel",[1],10),("Mortise Exit Device",920,"Stainless steel",[1],8),("Surface Vertical Rod Exit Device",1050,"Stainless steel",[1],7),("Exit Device Trim",165,"Stainless steel",[1],8)]),
 ("Faucets","FC",[("Single-Handle Lavatory Faucet",210,"Brass",[None],12),("Pull-Down Kitchen Faucet",385,"Brass",[None],10),("Touchless Sensor Faucet",520,"Brass / electronics",[None],10),("Metering Lavatory Faucet",290,"Brass",[None],6)]),
 ("Grab Bars","GB",[("18in Grab Bar",58,"Stainless steel",[None],8),("24in Grab Bar",66,"Stainless steel",[None],8),("36in Grab Bar",79,"Stainless steel",[None],8),("L-Shaped Grab Bar",135,"Stainless steel",[None],6)]),
 ("Bath Accessories","BA",[("Towel Bar 24in",48,"Zinc / brass",[None],8),("Toilet Paper Holder",32,"Zinc",[None],8),("Soap Dispenser",74,"Stainless steel",[None],8),("Framed Mirror 24x36",160,"Stainless steel / glass",[None],6)]),
]
rows = []; n = 1000
for cat, code, subs in CATS:
    for sub, base, mat, grades, cnt in subs:
        for i in range(cnt):
            n += 1
            fin_code, fin = FINISHES[rng.integers(0, len(FINISHES))] if code not in ("EX","DC","HG") or rng.random() < 0.5 else FINISHES[rng.integers(0, 2)]
            line = LINES[rng.integers(0, len(LINES))]
            g = grades[rng.integers(0, len(grades))]
            price = round(base * float(rng.uniform(0.85, 1.25)) * (1.12 if fin_code in ("BLK","US4") else 1.0), 2)
            # margin at LIST price; contractors buy at ~30% off list, so realized gross margin lands ~25-35%
            margin = float(rng.uniform(0.48, 0.58)) if code != "EX" else float(rng.uniform(0.54, 0.62))
            sku = f"TH-{code}-{n}"
            gtxt = f"ANSI/BHMA Grade {g}" if g else ("CSA B125.1 certified" if code == "FC" else "CSA B651 accessible design")
            rows.append(dict(sku=sku, name=f"{line} {sub} - {fin}", category=cat, subcategory=sub, product_line=line, finish=fin, finish_code=fin_code,
                             material=mat, ansi_grade=(str(g) if g else None), list_price_cad=price, cost_cad=round(price * (1 - margin), 2), image_path=None,
                             short_description=f"{line} series {sub.lower()} in {fin.lower()} ({fin_code}). {mat}. {gtxt}.",
                             launch_date=str(START - dt.timedelta(days=int(rng.integers(30, 2500)))), is_active=bool(rng.random() > 0.03)))
products = pd.DataFrame(rows)
img_skus = products.groupby("category").head(8).sku.tolist()      # 64 SKUs get a product image
products.loc[products.sku.isin(img_skus), "image_path"] = products.sku.map(lambda s: f"{VOL}/product_images/{s}.png")

SEG = [("General Contractor",0.45),("Condo Developer",0.25),("Property Manager",0.30)]
REG = [("BC",0.45),("ON",0.40),("US",0.15)]
CITY = {"BC":["Vancouver","Burnaby","Surrey","Richmond","Kelowna","Victoria","Coquitlam","North Vancouver"],
        "ON":["Toronto","Mississauga","Ottawa","Hamilton","Markham","Vaughan","London","Oakville"],
        "US":["Seattle, WA","Bellevue, WA","Portland, OR","Buffalo, NY","Detroit, MI","Spokane, WA"]}
SUFFIX = {"General Contractor":["Construction Ltd.","Builders Inc.","Contracting","General Contractors"],
          "Condo Developer":["Developments","Properties Group","Homes Corp.","Living"],
          "Property Manager":["Property Management","Residential Services","Realty Management","Strata Services"]}
REGION_RAW = {"BC":["BC","B.C.","British Columbia","bc"],"ON":["ON","Ont.","Ontario","ON "],"US":["US","USA","United States"]}
reps = {"BC":["Priya Sandhu","Mike Chen","Jordan Fraser"],"ON":["Alexandra Rossi","Daniel Okafor","Sophie Tremblay"],"US":["Kevin Walsh"]}
crow = []
for i in range(500):
    seg = random.choices([s for s,_ in SEG],[w for _,w in SEG])[0]; reg = random.choices([r for r,_ in REG],[w for _,w in REG])[0]
    city = random.choice(CITY[reg])
    nm = f"{fake.last_name()} {random.choice(SUFFIX[seg])}" if rng.random() < 0.6 else f"{fake.city()} {random.choice(SUFFIX[seg])}"
    size = float(rng.lognormal(0, 0.9)) * (2.2 if seg == "Condo Developer" else 1.0)
    crow.append(dict(customer_id=f"C{10001+i}", customer_name=nm, segment=seg, region_raw=random.choice(REGION_RAW[reg]), city=city,
                     country=("USA" if reg == "US" else "Canada"), account_manager=random.choice(reps[reg]),
                     payment_terms=random.choice(["Net 30","Net 30","Net 45","Net 60"]), credit_limit_cad=float(round(size*50000, -3) + 10000),
                     created_date=str(START - dt.timedelta(days=int(rng.integers(0, 3000)))), _size=size, _reg=reg))
customers = pd.DataFrame(crow)
print(len(products), "products,", len(customers), "customers")

In [0]:
SEAS = {1:.70,2:.78,3:.95,4:1.10,5:1.22,6:1.25,7:1.18,8:1.15,9:1.10,10:1.0,11:.88,12:.60}   # construction seasonality
def wh_for(c):
    if c["_reg"] == "US": return "Delta BC" if c["city"].endswith(("WA","OR")) else "Toronto"
    return {"BC":"Delta BC","ON":"Toronto"}[c["_reg"]]
cust = customers.to_dict("records"); cw = np.array([c["_size"] for c in cust]); cw = cw / cw.sum()
cat_list = [c for c,_,_ in CATS]
MIX = {"General Contractor":[.15,.17,.13,.14,.10,.11,.10,.10],
       "Condo Developer":   [.20,.22,.08,.18,.05,.12,.04,.11],
       "Property Manager":  [.12,.10,.17,.08,.08,.18,.15,.12]}
QTY = {"General Contractor":6,"Condo Developer":18,"Property Manager":3}
DISC = {"General Contractor":.27,"Condo Developer":.33,"Property Manager":.18}
psku = {c: products[products.category == c].to_dict("records") for c in cat_list}
orders = []; lines = []; oid = 100000; d = START
while d <= TODAY:
    yrs = (d - START).days / 365
    lam = 24 * SEAS[d.month] * (1 + 0.08*yrs) * (0.12 if d.weekday() >= 5 else 1.0)
    for _ in range(rng.poisson(lam)):
        oid += 1; c = cust[rng.choice(len(cust), p=cw)]; seg = c["segment"]; wh = wh_for(c)
        cancelled = rng.random() < 0.015
        mix = np.array(MIX[seg], dtype=float)
        if c["_reg"] == "ON" and d >= dt.date(2026,4,1): mix[4] *= 0.45     # STORY: exit devices losing ON share (price gap vs NorthBuild)
        mix /= mix.sum()
        orders.append(dict(order_id=f"SO-{oid}", customer_id=c["customer_id"],
                           order_date=d.strftime("%m/%d/%Y") if rng.random() < 0.03 else d.isoformat(),   # messy: mixed date formats
                           ship_date=None if cancelled else (d + dt.timedelta(days=int(rng.integers(1,6)))).isoformat(),
                           warehouse=wh, sales_rep=c["account_manager"], channel=random.choices(["Inside Sales","Field Sales","Web Portal","EDI"],[.35,.30,.25,.10])[0],
                           status="Cancelled" if cancelled else "Shipped", po_number=f"PO-{rng.integers(10000,99999)}",
                           freight_cad=float(round(rng.uniform(25,350),2)), currency="CAD"))
        for ln in range(1, 2 + rng.poisson(1.6)):
            cat = cat_list[rng.choice(8, p=mix)]; p = psku[cat][rng.integers(0, len(psku[cat]))]
            q = max(1, int(rng.lognormal(np.log(QTY[seg]), 0.7)))
            disc = float(np.clip(rng.normal(DISC[seg], 0.04) + (0.03 if q > 30 else 0), 0.05, 0.48))
            bo_p = 0.40 if (wh == "Toronto" and cat == "Door Closers" and d >= dt.date(2026,8,1)) else 0.035   # STORY: Toronto closer supply delay
            bo = 0 if cancelled or rng.random() > bo_p else max(1, int(q * rng.uniform(0.3, 1.0)))
            shipped = 0 if cancelled else q - bo
            ret_p = 0.02 + (0.08 if (cat == "Faucets" and p["finish_code"] == "BLK") else 0)                # STORY: Matte Black faucet returns
            ret = 0 if shipped == 0 or rng.random() > ret_p else max(1, int(shipped * rng.uniform(0.1, 0.5)))
            sku = p["sku"]; r = rng.random()                                                                 # messy SKU formats
            if r < 0.015: sku = sku.lower()
            elif r < 0.03: sku = sku.replace("-", " ")
            elif r < 0.04: sku = sku.replace("-", "")
            row = dict(order_id=f"SO-{oid}", line_no=ln, sku=sku, qty_ordered=q, qty_shipped=shipped, qty_backordered=bo, qty_returned=ret,
                       unit_list_price=p["list_price_cad"], discount_pct=round(disc,4), unit_net_price=round(p["list_price_cad"]*(1-disc),2), unit_cost=p["cost_cad"])
            lines.append(row)
            if rng.random() < 0.008: lines.append(dict(row))   # messy: duplicate from source re-sync
    d += dt.timedelta(days=1)
orders_df = pd.DataFrame(orders); lines_df = pd.DataFrame(lines)
ld = lines_df.merge(orders_df[["order_id","order_date","warehouse","customer_id","status"]], on="order_id")
print(len(orders_df), "orders,", len(lines_df), "lines")

In [0]:
sku_fix = {s.replace("-",""): s for s in products.sku}
ld["sku_c"] = ld.sku.str.upper().str.replace(r"[^A-Z0-9]", "", regex=True).map(sku_fix)
ld["od"] = pd.to_datetime(ld.order_date, format="mixed")
pmap = products.set_index("sku").to_dict("index")

# --- inventory snapshot by warehouse ---
dem = ld[ld.od >= pd.Timestamp(TODAY - dt.timedelta(days=90))].groupby(["warehouse","sku_c"]).qty_ordered.sum() / 90
inv = []
for wh in ["Delta BC","Toronto"]:
    for p in products.to_dict("records"):
        adu = float(dem.get((wh, p["sku"]), 0.0)); ss = int(np.ceil(adu*14)); rop = int(np.ceil(adu*30)) + ss
        oh = int(max(0, adu * rng.uniform(20, 75)))
        if wh == "Toronto" and p["category"] == "Door Closers": oh = int(adu * rng.uniform(0, 6))   # STORY: supplier delay
        alloc = int(min(oh, adu * rng.uniform(2, 8)))
        inv.append(dict(warehouse=wh, sku=p["sku"], on_hand_qty=oh, allocated_qty=alloc, on_order_qty=int(max(0, rop*rng.uniform(0.5,2.0))) if oh < rop else 0,
                        reorder_point=rop, safety_stock=ss, bin_location=f"{'D' if wh=='Delta BC' else 'T'}-{rng.integers(1,30):02d}-{rng.integers(1,12):02d}",
                        snapshot_date=TODAY.isoformat(), last_receipt_date=(TODAY - dt.timedelta(days=int(rng.integers(3,120)))).isoformat()))
inventory = pd.DataFrame(inv)

# --- competitor prices: 1-2 equivalents per SKU, monthly scrapes for 6 months ---
COMPS = {"NorthBuild Supply":("NB","northbuildsupply.example"),"Pacific Hardware Co":("PHC","pacifichardware.example"),"DoorPro Direct":("DPD","doorprodirect.example")}
FACT = {"NorthBuild Supply":{"Exit Devices":.83,"Door Closers":.92,"default":.98},          # STORY: NorthBuild undercuts exit devices
        "Pacific Hardware Co":{"Faucets":.95,"Bath Accessories":.93,"default":1.03},
        "DoorPro Direct":{"Exit Devices":.88,"Locks":.96,"Levers":.97,"default":1.04}}
snaps = sorted(TODAY.replace(day=1) - pd.DateOffset(months=m) for m in range(6))
cp = []
for p in products.to_dict("records"):
    for cn in random.sample(list(COMPS), k=int(rng.integers(1,3))):
        pre, dom = COMPS[cn]; csku = f"{pre}-{rng.integers(10000,99999)}"
        base = FACT[cn].get(p["category"], FACT[cn]["default"]) * float(rng.normal(1, 0.045))
        for i, sd in enumerate(snaps):
            drift = 1 + 0.004*i*(1 if rng.random() < 0.5 else -1)
            cp.append(dict(competitor_sku=csku, competitor_name=cn, t_sku=p["sku"], competitor_product_name=f"{p['subcategory']} {p['finish_code']}",
                           competitor_price_cad=round(p["list_price_cad"]*base*drift, 2), price_date=(sd.date() + dt.timedelta(days=int(rng.integers(0,5)))).isoformat(),
                           source_url=f"https://www.{dom}/p/{csku.lower()}", in_stock=bool(rng.random() > 0.1)))
competitor_prices = pd.DataFrame(cp)

# --- HubSpot companies (one per ERP customer + 40 prospects) ---
hs_c = []
for i, c in enumerate(cust):
    hs_c.append(dict(company_id=int(7000000+i*7), name=c["customer_name"], domain=c["customer_name"].lower().replace(" ","").replace(".","")[:20]+".example",
                     industry={"General Contractor":"Construction","Condo Developer":"Real Estate","Property Manager":"Real Estate"}[c["segment"]],
                     city=c["city"], state_region=c["region_raw"], lifecycle_stage="customer", erp_customer_id=c["customer_id"], hubspot_owner=c["account_manager"],
                     createdate=c["created_date"]+"T09:00:00Z"))
for j in range(40):
    reg = random.choice(["BC","ON"])
    hs_c.append(dict(company_id=int(7900000+j), name=f"{fake.last_name()} {random.choice(SUFFIX['General Contractor'])}", domain=None, industry="Construction",
                     city=random.choice(CITY[reg]), state_region=reg, lifecycle_stage=random.choice(["lead","opportunity"]), erp_customer_id=None,
                     hubspot_owner=random.choice(reps[reg]), createdate=(TODAY - dt.timedelta(days=int(rng.integers(5,300)))).isoformat()+"T09:00:00Z"))
hubspot_companies = pd.DataFrame(hs_c)
cid2hs = {c["erp_customer_id"]: c["company_id"] for c in hs_c if c["erp_customer_id"]}

# --- HubSpot deals: closed-won deals linked to the 600 largest ERP orders, plus open/lost deals ---
ov = ld.assign(v=ld.qty_ordered*ld.unit_net_price).groupby(["order_id","customer_id","order_date"]).v.sum().reset_index().sort_values("v", ascending=False).head(600)
PROJ = ["condo tower","townhouse complex","seniors residence","school retrofit","office fit-out","hospital wing","rental building","hotel renovation"]
deals = []
for k, o in enumerate(ov.itertuples()):
    od = pd.to_datetime(o.order_date, format="mixed").date()
    deals.append(dict(deal_id=int(9100000+k), company_id=cid2hs[o.customer_id], dealname=f"{random.choice(PROJ).title()} - {rng.integers(40,400)} units", pipeline="Projects",
                      dealstage="closedwon", amount=float(round(o.v,2)), closedate=(od - dt.timedelta(days=int(rng.integers(3,30)))).isoformat(), erp_order_id=o.order_id,
                      hubspot_owner=None, createdate=(od - dt.timedelta(days=int(rng.integers(30,120)))).isoformat()))
for k in range(250):
    c = cust[rng.integers(0, len(cust))]; cd = TODAY - dt.timedelta(days=int(rng.integers(-60,600)))
    deals.append(dict(deal_id=int(9200000+k), company_id=cid2hs[c["customer_id"]], dealname=f"{random.choice(PROJ).title()} - {rng.integers(40,400)} units", pipeline="Projects",
                      dealstage=random.choices(["closedlost","contractsent","qualifiedtobuy","presentationscheduled"],[.45,.2,.2,.15])[0],
                      amount=float(round(rng.lognormal(10.5,0.8),2)), closedate=cd.isoformat(), erp_order_id=None, hubspot_owner=c["account_manager"],
                      createdate=(cd - dt.timedelta(days=int(rng.integers(30,120)))).isoformat()))
hubspot_deals = pd.DataFrame(deals)

# --- HubSpot tickets: free text (themes are classified later with ai_classify in silver) ---
T = {
 "Finish defect":["Finish peeling on {name}","Customer reports the {fin} finish on {sku} is flaking/discolouring after {m} months. Units installed in washrooms at {proj}. Requesting replacement under finish warranty. Photos attached."],
 "Backorder / late delivery":["Backorder ETA for {sku} on {so}?","We are still waiting on {q} units of {sku} ({name}) from order {so}. Site needs them for inspection next week. Please confirm ETA from {wh}."],
 "Wrong item shipped":["Wrong finish shipped on {so}","Received {sku} but the PO called for a different finish. Need exchange; units are unopened on site."],
 "Damaged in transit":["Damaged carton - {sku}","{q} units of {name} arrived with crushed cartons and bent arms. Freight carrier signed as damaged. Photos of the boxes attached."],
 "Installation question":["Install question: {name}","Installer asking for the template / spring size adjustment for {sku} on a 36in door with parallel arm mount. Do we have the install SOP?"],
 "Warranty claim":["Warranty claim {sku}","{name} failing to latch after {m} months in a high-traffic corridor. Requesting warranty replacement. Order {so}."],
 "Pricing / invoice dispute":["Invoice discrepancy on {so}","Customer says the quoted project price for {sku} was lower than invoiced. A competitor quoted less; asking us to honour the quote."],
}
ol = ld[ld.status == "Shipped"].copy()
ol["pcat"] = ol.sku_c.map(lambda s: pmap[s]["category"]); ol["fin"] = ol.sku_c.map(lambda s: pmap[s]["finish_code"])
tickets = []
for i in range(620):
    r = ol.sample(1, random_state=int(rng.integers(0, 1e9))).iloc[0]
    th = random.choices(list(T), [.12,.18,.10,.10,.16,.14,.20])[0]
    if r.pcat == "Faucets" and r.fin == "BLK" and rng.random() < 0.6: th = "Finish defect"
    if r.pcat == "Door Closers" and r.warehouse == "Toronto" and r.od >= pd.Timestamp("2026-08-01") and rng.random() < 0.7: th = "Backorder / late delivery"
    if r.pcat == "Exit Devices" and rng.random() < 0.35: th = "Pricing / invoice dispute"
    p = pmap[r.sku_c]; created = (r.od + pd.Timedelta(days=int(rng.integers(3,90)))).to_pydatetime()
    if created.date() > TODAY: created = dt.datetime.combine(TODAY, dt.time(10))
    f = dict(name=p["name"], sku=r.sku_c, fin=p["finish"], so=r.order_id, q=int(r.qty_ordered), m=int(rng.integers(2,18)), wh=r.warehouse,
             proj=random.choice(["the Burnaby tower","a Mississauga townhouse site","a Surrey rental building","the Ottawa school retrofit"]))
    subj, body = [s.format(**f) for s in T[th]]
    link = rng.random()
    closed = created + dt.timedelta(days=float(rng.uniform(0.5,20))) if rng.random() < 0.8 and created.date() < TODAY - dt.timedelta(days=3) else None
    tickets.append(dict(ticket_id=int(55000000+i), company_id=cid2hs[r.customer_id], subject=subj, content=body,
                        hs_pipeline_stage="Closed" if closed else random.choice(["New","Waiting on contact","Waiting on us"]),
                        hs_ticket_priority=random.choices(["LOW","MEDIUM","HIGH"],[.3,.5,.2])[0],
                        hs_ticket_category=random.choice([th.upper().replace(" / ","_").replace(" ","_"), None, "PRODUCT_ISSUE", "GENERAL_INQUIRY"]),
                        erp_order_number=(r.order_id if link < 0.7 else (r.order_id.replace("SO-","so ") if link < 0.8 else None)),   # messy order references
                        sku_mentioned=r.sku_c if rng.random() < 0.8 else None,
                        createdate=created.isoformat()+"Z", closed_date=(closed.isoformat()+"Z") if closed else None))
hubspot_tickets = pd.DataFrame(tickets)
print(len(inventory), "inventory rows,", len(competitor_prices), "competitor prices,", len(hubspot_companies), "companies,", len(hubspot_deals), "deals,", len(hubspot_tickets), "tickets")

In [0]:
def save(pdf_, name, comment, cols):
    """Overwrite a bronze Delta table and document it (table + every column)."""
    full = f"{B}.bronze_{name}"
    (spark.createDataFrame(pdf_.astype(object).where(pdf_.notna(), None))
          .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(full))
    spark.sql(f"COMMENT ON TABLE {full} IS {comment!r}")
    for c, txt in cols.items():
        spark.sql(f"ALTER TABLE {full} ALTER COLUMN `{c}` COMMENT {txt!r}")

save(products, "products", "ERP item master (raw). One row per T Industries SKU: door hardware (locks, levers, door closers, hinges, exit devices) and bathroom fixtures (faucets, grab bars, accessories).", dict(
  sku="T Industries SKU, format TH-<category code>-<number>", name="Product display name (line + subtype + finish)", category="Product category", subcategory="Product subtype",
  product_line="Design collection / product line", finish="Finish name", finish_code="BHMA finish code (US26D, US32D, BLK, US4, US10B, US26)", material="Primary material",
  ansi_grade="ANSI/BHMA grade 1-3 (1 = heaviest duty); null for bathroom fixtures", list_price_cad="National list price, CAD", cost_cad="Standard unit cost, CAD",
  image_path="UC volume path of the product image, if one exists", short_description="Short marketing description", launch_date="Date the SKU was launched (string)", is_active="SKU currently sellable"))
save(customers.drop(columns=["_size","_reg"]), "customers", "ERP customer master (raw): general contractors, condo developers and property managers in BC, Ontario and the US. region_raw has inconsistent spellings.", dict(
  customer_id="ERP customer id", customer_name="Customer legal / trade name", segment="Customer segment", region_raw="Province / country as keyed in the ERP (inconsistent)",
  city="City (US cities include state)", country="Country", account_manager="Assigned account manager", payment_terms="Payment terms", credit_limit_cad="Credit limit, CAD", created_date="Account creation date (string)"))
save(orders_df, "sales_orders", "ERP sales order headers (raw), 24 months. order_date has mixed formats (yyyy-MM-dd and MM/dd/yyyy).", dict(
  order_id="Sales order number SO-nnnnnn", customer_id="ERP customer id", order_date="Order date (string, mixed formats)", ship_date="Ship date (string), null if cancelled",
  warehouse="Fulfilling warehouse: Delta BC or Toronto", sales_rep="Sales rep on the order", channel="Order channel", status="Shipped or Cancelled", po_number="Customer PO number",
  freight_cad="Freight charged, CAD (excluded from net sales)", currency="Invoice currency (all CAD)"))
save(lines_df, "order_lines", "ERP sales order lines (raw). Contains ~1% duplicate rows from source re-syncs and inconsistent SKU formatting.", dict(
  order_id="Sales order number", line_no="Line number within the order", sku="SKU as keyed (inconsistent case / separators)", qty_ordered="Units ordered", qty_shipped="Units shipped",
  qty_backordered="Units backordered (not shipped on the original ship date)", qty_returned="Units later returned (credit memo)", unit_list_price="List price per unit at order time, CAD",
  discount_pct="Discount off list (0-1)", unit_net_price="Net invoiced price per unit, CAD", unit_cost="Standard cost per unit, CAD"))
save(inventory, "inventory", "WMS inventory snapshot by warehouse (Delta BC, Toronto) and SKU.", dict(
  warehouse="Warehouse", sku="SKU", on_hand_qty="Units on hand", allocated_qty="Units allocated to open orders", on_order_qty="Units on open purchase orders",
  reorder_point="Reorder point, units", safety_stock="Safety stock, units", bin_location="Bin location", snapshot_date="Snapshot date (string)", last_receipt_date="Last receipt date (string)"))
save(competitor_prices, "competitor_prices", "Competitor web prices scraped monthly (fictional competitors NorthBuild Supply, Pacific Hardware Co, DoorPro Direct) mapped to equivalent T SKUs.", dict(
  competitor_sku="Competitor's SKU", competitor_name="Competitor", t_sku="Equivalent T Industries SKU", competitor_product_name="Competitor product description",
  competitor_price_cad="Competitor web price, CAD", price_date="Scrape date (string)", source_url="Product page URL", in_stock="Competitor showed the item in stock"))
save(hubspot_companies, "hubspot_companies", "HubSpot CRM companies (Lakeflow Connect HubSpot connector shape). erp_customer_id links to the ERP customer.", dict(
  company_id="HubSpot company id", name="Company name", domain="Web domain", industry="Industry", city="City", state_region="Province/state as entered in HubSpot",
  lifecycle_stage="HubSpot lifecycle stage", erp_customer_id="ERP customer id (custom property); null for prospects", hubspot_owner="Owner", createdate="Created timestamp (ISO string)"))
save(hubspot_deals, "hubspot_deals", "HubSpot project deals. Closed-won deals carry erp_order_id linking to the ERP sales order.", dict(
  deal_id="HubSpot deal id", company_id="HubSpot company id", dealname="Deal name", pipeline="Deal pipeline", dealstage="Deal stage", amount="Deal amount, CAD",
  closedate="Close date (string)", erp_order_id="Linked ERP sales order (closed-won only)", hubspot_owner="Owner", createdate="Created date (string)"))
save(hubspot_tickets, "hubspot_tickets", "HubSpot service tickets. erp_order_number is free text (SO-nnnnnn, 'so nnnnnn' or missing - then the order may only be in the subject).", dict(
  ticket_id="HubSpot ticket id", company_id="HubSpot company id", subject="Ticket subject", content="Ticket description", hs_pipeline_stage="Ticket stage",
  hs_ticket_priority="Priority", hs_ticket_category="Category picked by the agent (inconsistent, often blank)", erp_order_number="ERP order reference as typed",
  sku_mentioned="SKU referenced on the ticket", createdate="Created timestamp (ISO string)", closed_date="Closed timestamp (ISO string)"))
print("structured bronze written")

In [0]:
from PIL import Image, ImageDraw, ImageFont
FCOL = {"US26D":(185,188,192),"US32D":(170,172,170),"BLK":(35,35,38),"US4":(196,160,90),"US10B":(70,48,35),"US26":(215,220,228)}
def font(sz):
    for f in ["/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf","/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"]:
        if os.path.exists(f): return ImageFont.truetype(f, sz)
    return ImageFont.load_default()
def draw_product(d, cat, col):
    R = lambda a,b,c,e,**k: d.rounded_rectangle([(a,b),(c,e)], radius=12, **k); E = lambda a,b,c,e,**k: d.ellipse([(a,b),(c,e)], **k); out = (60,60,60)
    if cat == "Levers": R(260,150,340,250,fill=col,outline=out,width=3); E(240,170,360,290,fill=col,outline=out,width=3); R(300,210,520,250,fill=col,outline=out,width=3)
    elif cat == "Locks": R(220,110,380,330,fill=col,outline=out,width=3); E(265,160,335,230,fill=(90,90,90)); R(292,195,308,280,fill=(30,30,30))
    elif cat == "Door Closers": R(110,170,490,260,fill=col,outline=out,width=3); d.line([(380,250),(470,330),(250,330)],fill=out,width=10); E(370,240,395,265,fill=out)
    elif cat == "Hinges": R(180,100,295,340,fill=col,outline=out,width=3); R(305,100,420,340,fill=col,outline=out,width=3); R(288,90,312,350,fill=(120,120,120))
    elif cat == "Exit Devices": R(70,190,530,250,fill=col,outline=out,width=3); R(90,205,420,235,fill=(min(255,col[0]+30),)*3,outline=out); R(440,170,520,270,fill=col,outline=out,width=3)
    elif cat == "Faucets": R(270,230,330,340,fill=col,outline=out,width=3); d.arc([(220,110),(420,330)],180,360,fill=col,width=28); R(395,200,425,240,fill=col); R(250,330,350,350,fill=col,outline=out)
    elif cat == "Grab Bars": R(90,200,510,240,fill=col,outline=out,width=3); E(70,180,130,260,fill=col,outline=out,width=3); E(470,180,530,260,fill=col,outline=out,width=3)
    else: R(120,190,480,220,fill=col,outline=out,width=3); R(130,170,160,240,fill=col,outline=out); R(440,170,470,240,fill=col,outline=out)
def product_image(p, path, photo=False):
    img = Image.new("RGB", (600,600), (205,196,182) if photo else (242,240,236)); d = ImageDraw.Draw(img)
    if photo:   # rough "site photo": wood-grain door background
        for y in range(0, 600, 9): d.line([(0,y),(600,y+int(rng.integers(-6,6)))], fill=(190+int(rng.integers(-12,12)),170,140), width=4)
    draw_product(d, p["category"], FCOL[p["finish_code"]])
    d.rectangle([(40,420),(560,570)], fill=(255,255,255), outline=(20,40,90), width=4)        # packaging label
    d.text((60,432), "T INDUSTRIES", font=font(26), fill=(20,40,90)); d.text((60,470), p["sku"], font=font(34), fill=(0,0,0))
    d.text((60,515), p["name"][:44], font=font(18), fill=(40,40,40))
    d.text((60,540), f"Finish {p['finish_code']}  |  " + (f"ANSI Grade {p['ansi_grade']}" if p["ansi_grade"] else p["category"]), font=font(16), fill=(80,80,80))
    img.save(path, "PNG")
os.makedirs(f"{VOL}/product_images", exist_ok=True); os.makedirs(f"{VOL}/photo_uploads", exist_ok=True)
for p in products[products.image_path.notna()].to_dict("records"): product_image(p, p["image_path"])
# photo_uploads/ holds hand-curated, real customer-style product photos (other brands, no T SKU) that the presenter
# uploads to Genie One during the demo. Nothing in the demo reads them programmatically, and the generator never writes them.
print(len(os.listdir(f"{VOL}/product_images")), "product images")

In [0]:
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors
import docx as _docx
SP = f"{VOL}/sharepoint"
if os.path.exists(SP): shutil.rmtree(SP)
for sub in ["Product Spec Sheets","Installation SOPs","Customer Service SOPs","HR","Business Rules","Pricing"]: os.makedirs(f"{SP}/{sub}", exist_ok=True)
ss = getSampleStyleSheet()

def _via_tmp(fn):
    """Volumes (FUSE) don't support random-access writes (zip/docx), so build locally and copy."""
    def w(path, *a, **k):
        tmp = os.path.join(tempfile.gettempdir(), os.path.basename(path)); fn(tmp, *a, **k); shutil.copyfile(tmp, path); os.remove(tmp)
    return w
@_via_tmp
def pdf(path, title, sections, meta="Document owner: Product Management | Revision C | T Industries Ltd., Delta, BC"):
    el = [Paragraph(title, ss["Title"]), Paragraph(meta, ss["Italic"]), Spacer(1,10)]
    for h, body in sections:
        el.append(Paragraph(h, ss["Heading2"]))
        if isinstance(body, list) and body and isinstance(body[0], list):
            t = Table(body, hAlign="LEFT"); t.setStyle(TableStyle([("GRID",(0,0),(-1,-1),0.5,colors.grey),("BACKGROUND",(0,0),(-1,0),colors.lightgrey),("FONTSIZE",(0,0),(-1,-1),9)])); el.append(t)
        else:
            for para in (body if isinstance(body, list) else [body]): el.append(Paragraph(para, ss["BodyText"]))
        el.append(Spacer(1,8))
    SimpleDocTemplate(path, pagesize=letter, title=title).build(el)
@_via_tmp
def docx_file(path, title, sections):
    d = _docx.Document(); d.add_heading(title, 0); d.add_paragraph("T Industries Ltd. - Controlled document. Printed copies are uncontrolled.")
    for h, body in sections:
        d.add_heading(h, 1)
        for para in (body if isinstance(body, list) else [body]):
            if para.startswith("#. "): d.add_paragraph(para[3:], style="List Number")
            elif para.startswith("- "): d.add_paragraph(para[2:], style="List Bullet")
            else: d.add_paragraph(para)
    d.save(path)
def spec(prefix, sub):
    ps = products[products.sku.str.startswith(prefix) & (products.subcategory == sub)].head(6)
    return [["SKU","Description","Finish","List CAD"]] + [[r.sku, r.name[:42], r.finish_code, f"{r.list_price_cad:,.2f}"] for r in ps.itertuples()]

# --- Product spec sheets (PDF) ---
pdf(f"{SP}/Product Spec Sheets/DC-Series Commercial Door Closer Spec Sheet.pdf", "DC-Series Commercial Surface Door Closer - Specification Sheet", [
 ("Overview", "The DC-Series surface door closer is a heavy-duty, non-handed hydraulic closer for commercial, institutional and multi-residential openings. Certified to ANSI/BHMA A156.4 Grade 1 (10,000,000 cycles) and UL 10C for 3-hour fire-rated doors."),
 ("Features", ["Adjustable spring power sizes 1-6; regular, top-jamb and parallel-arm mounting.", "Separate valves for sweep, latch and backcheck; optional delayed action.", "Barrier-free compliant at 5 lbf opening force for interior doors (CSA B651)."]),
 ("Models and pricing", spec("TH-DC", "Surface Door Closer")),
 ("Warranty", "Commercial door closers carry a 10-year limited warranty on the closer body and hydraulics, and 2 years on arms and accessories. Residential closers (Grade 3) carry a 2-year limited warranty. See Warranty Policy WP-01 for the claim procedure."),
 ("Ordering", "Specify model, arm type, finish (US26D, US32D, BLK, US4, US10B, US26) and mounting. Standard lead time 3-5 business days from Delta BC or Toronto.")])
pdf(f"{SP}/Product Spec Sheets/EX-Series Rim Exit Device Spec Sheet.pdf", "EX-Series Rim and Mortise Exit Devices - Specification Sheet", [
 ("Overview", "Grade 1 touch-bar exit devices for life-safety egress. Certified ANSI/BHMA A156.3 Grade 1, UL 305 panic hardware and UL 10C fire exit hardware (3 hours)."),
 ("Functions", "Exit only, night latch, passage, classroom, storeroom. Field-reversible. Fits doors 1-3/4in to 2-1/4in thick, 36in and 48in widths."),
 ("Models and pricing", spec("TH-EX", "Rim Exit Device")),
 ("Warranty", "Exit devices carry a 5-year limited mechanical warranty. Electrified options (latch retraction, request-to-exit) carry 2 years on electronic components."),
 ("Code notes", "Install per NFPA 101 / National Building Code of Canada 3.4.6.16. The touch bar must extend at least one-half of the door width.")])
pdf(f"{SP}/Product Spec Sheets/LK-3100 Mortise Lock Spec Sheet.pdf", "LK-3100 Grade 1 Mortise Lock - Specification Sheet", [
 ("Overview", "Heavy-duty mortise lock body with 30 functions for high-traffic commercial and institutional doors. ANSI/BHMA A156.13 Series 1000 Grade 1."),
 ("Models and pricing", spec("TH-LK", "Grade 1 Mortise Lock")),
 ("Warranty", "Commercial locks: 7-year limited mechanical warranty; finishes 3 years. Electronic keypad locks: 2 years on electronics, 5 years mechanical.")])
pdf(f"{SP}/Product Spec Sheets/Granville Lever Collection Spec Sheet.pdf", "Granville Lever Collection - Specification Sheet", [
 ("Overview", "Cylindrical lever sets in passage, privacy, storeroom and entry functions. Grades 1-3 depending on function. The Matte Black (BLK) finish uses a powder-coat process."),
 ("Models and pricing", spec("TH-LV", "Passage Lever Set")),
 ("Warranty", "Residential lever sets: limited lifetime mechanical warranty and 5-year finish warranty. Commercial lever sets: 7-year mechanical, 3-year finish.")])
pdf(f"{SP}/Product Spec Sheets/FC-Series Touchless Sensor Faucet Spec Sheet.pdf", "FC-Series Touchless Sensor Faucet - Specification Sheet", [
 ("Overview", "Deck-mounted touchless lavatory faucet, 1.9 L/min (0.5 gpm), CSA B125.1 / ASME A112.18.1 certified, battery or AC powered."),
 ("Models and pricing", spec("TH-FC", "Touchless Sensor Faucet")),
 ("Finish care", "Clean with mild soap and water only. Do not use abrasive or chlorine cleaners on Matte Black (BLK) or Satin Brass (US4) finishes; damage from harsh chemicals is excluded from the finish warranty."),
 ("Warranty", "Commercial faucets: 5-year limited warranty on the valve body, 2 years on sensors and solenoids, and 1 year on the BLK finish in commercial washrooms. Residential faucets: limited lifetime on the cartridge.")])
pdf(f"{SP}/Product Spec Sheets/GB-Series Grab Bar Spec Sheet.pdf", "GB-Series Stainless Grab Bars - Specification Sheet", [
 ("Overview", "18-gauge type 304 stainless steel grab bars, 1-1/4in diameter, 1-1/2in wall clearance. Supports 250 lbf (1112 N) when installed per instructions; meets CSA B651 and ADA 609."),
 ("Models and pricing", spec("TH-GB", "24in Grab Bar")),
 ("Warranty", "Lifetime limited warranty against manufacturing defects in residential use; 10 years commercial.")])
# --- Customer service SOPs ---
pdf(f"{SP}/Customer Service SOPs/WP-01 Warranty Policy.pdf", "WP-01 Product Warranty Policy", [
 ("Purpose", "Defines warranty periods and the claim process for all T Industries door hardware and bathroom fixtures sold in Canada and the United States."),
 ("Warranty periods", [["Product family","Mechanical / body","Finish","Electronics"],
   ["Commercial door closers (Grade 1)","10 years","3 years","n/a"],["Residential door closers (Grade 3)","2 years","1 year","n/a"],
   ["Exit devices","5 years","3 years","2 years"],["Commercial locks and levers","7 years","3 years","2 years"],
   ["Residential locks and levers","Limited lifetime","5 years","2 years"],["Hinges and pivots","Lifetime (continuous hinges 10 yrs)","3 years","n/a"],
   ["Commercial faucets","5 years","1 year (BLK) / 3 years other","2 years"],["Grab bars","10 years commercial / lifetime residential","3 years","n/a"],
   ["Bath accessories","2 years","1 year","n/a"]]),
 ("Claim procedure", ["1. Customer opens a ticket in HubSpot with the order number (SO-xxxxxx), SKU, quantity and photos of the defect.",
   "2. Customer Service validates the purchase date against the ERP order and confirms the product is within its warranty period.",
   "3. Approved claims receive an RMA number; the replacement ships from the nearest warehouse (Delta BC or Toronto) within 2 business days.",
   "4. Finish claims require photos showing the defect and confirmation that approved cleaning products were used."]),
 ("Exclusions", "Improper installation, abuse, harsh or abrasive cleaners, modifications, normal wear of finishes in coastal/salt-air environments beyond 1 year, and labour costs.")],
 meta="Document owner: Customer Service | Effective 2026-01-01 | Revision F")
docx_file(f"{SP}/Customer Service SOPs/CS-04 Returns and RMA SOP.docx", "CS-04 Returns and RMA Standard Operating Procedure", [
 ("Scope", "Applies to all returns from contractors, developers, property managers and dealers in BC, Ontario and the US."),
 ("Return eligibility", ["- Stock items in original, unopened packaging may be returned within 30 days of the ship date.", "- A 20% restocking fee applies to non-defective returns; it is waived for T Industries shipping errors.",
   "- Special-order, custom-finish and keyed/cored items are non-returnable.", "- Returns older than 90 days require Sales Manager approval."]),
 ("Procedure", ["#. Customer Service opens or updates the HubSpot ticket and records the ERP order number.", "#. Issue an RMA number in the ERP and email the RMA form to the customer.",
   "#. Customer ships the goods to the originating warehouse (Delta BC or Toronto) with the RMA number on the carton.",
   "#. Warehouse inspects within 2 business days and records the condition (A = resaleable, B = damaged, C = defective).",
   "#. A credit memo is issued for the net invoiced price less any restocking fee; returns reduce net sales in the month the credit is issued."]),
 ("Damaged in transit", "Customers must note damage on the carrier's delivery receipt. Customer Service files the freight claim; the replacement ships immediately at no charge.")])
# --- Installation SOPs (DOCX) ---
docx_file(f"{SP}/Installation SOPs/INST-DC Surface Door Closer Installation.docx", "INST-DC Surface Door Closer Installation Procedure", [
 ("Tools required", ["- Drill with 7/32in and 1/4in bits, #3 Phillips, 5/32in hex key, level, closer template."]),
 ("Installation steps", ["#. Identify the mounting type (regular arm, top jamb or parallel arm) and the door hand.", "#. Tape the paper template to the door and frame; mark and drill pilot holes.",
   "#. Mount the closer body with the regulating valves facing the hinge side (regular arm) or away from it (parallel arm).", "#. Attach the arm shoe to the frame (or the parallel-arm bracket to the soffit).",
   "#. Pre-load the spring by rotating the main arm 45 degrees and connect the forearm with the door closed.",
   "#. Set spring power to the door width: size 3 for 36in interior doors, size 4-5 for exterior or 42-48in doors.",
   "#. Adjust sweep speed (valve S) to 3-7 seconds from 90 to 10 degrees, then latch speed (valve L) and backcheck (valve BC).",
   "#. Install the cover and verify the opening force is 5 lbf or less for barrier-free interior doors."]),
 ("Troubleshooting", "If the door slams, turn valve L clockwise 1/8 turn. Never back a valve out more than 2 full turns - hydraulic fluid will leak and void the warranty.")])
docx_file(f"{SP}/Installation SOPs/INST-EX Rim Exit Device Installation.docx", "INST-EX Rim Exit Device Installation Procedure", [
 ("Before you start", "Confirm door thickness, width and that the strike will engage the frame. Cut the device to length if needed (never cut the active case)."),
 ("Installation steps", ["#. Mark the horizontal centreline 39-1/16in above the finished floor.", "#. Position the template and drill the mounting holes; through-bolt on hollow metal and wood doors.",
   "#. Cut the touch bar to the door width minus 2in and install the end cap.", "#. Mount the active case and end cap; install the rim strike and adjust so the latch projects fully.",
   "#. Install outside trim and cylinder (if specified) and test dogging and latch retraction.", "#. On fire-rated doors do not install dogging - verify the device is labelled as fire exit hardware."])])
docx_file(f"{SP}/Installation SOPs/INST-GB Grab Bar Installation.docx", "INST-GB Grab Bar Installation Procedure", [
 ("Requirements", "Install into solid blocking or with approved toggle anchors. Height 840-920 mm (33-36 in) above the finished floor per CSA B651."),
 ("Installation steps", ["#. Locate blocking or studs; mark flange positions with a level.", "#. Drill and install anchors rated for at least 250 lbf per flange.",
   "#. Fasten flanges with the supplied stainless screws and snap on the covers.", "#. Load-test the bar by hand and record the installation in the unit turnover checklist."])])
docx_file(f"{SP}/Installation SOPs/INST-LK Mortise Lock Installation.docx", "INST-LK Mortise Lock Installation Procedure", [
 ("Installation steps", ["#. Mortise the door edge to 1in x 6in x 4-7/8in deep using the template.", "#. Drill through-holes for the lever spindle and cylinder.",
   "#. Insert the lock body and set the handing by reversing the latch bolt.", "#. Install the cylinder and tighten the set screw; install levers and roses.",
   "#. Install the strike on the frame and confirm a deadbolt throw of 1in."])])
# --- Pricing ---
docx_file(f"{SP}/Pricing/Pricing and Discount Authority Matrix.docx", "Pricing and Discount Authority Matrix (FY2026)", [
 ("Price lists", "List prices are set nationally in CAD and reviewed quarterly against competitor web pricing (NorthBuild Supply, Pacific Hardware Co, DoorPro Direct)."),
 ("Discount authority", ["- Inside / field sales rep: up to 30% off list.", "- Regional sales manager: up to 38% off list.", "- VP Sales: above 38% off list, or any price below cost + 15%.",
   "- Project (HubSpot deal) pricing is valid for 90 days from the quote date."]),
 ("Competitive response", "If a SKU's list price is more than 10% above the lowest competitor price for 2 consecutive months, Product Management reviews the price at the next quarterly review.")])
# --- HR excerpt ---
pdf(f"{SP}/HR/Employee Handbook - Canada (Excerpt).pdf", "Employee Handbook - Canada (Excerpt)", [
 ("Vacation", "Employees in British Columbia receive 2 weeks paid vacation after 1 year and 3 weeks after 5 years of service, per the BC Employment Standards Act. Ontario employees receive 2 weeks after 1 year and 3 weeks after 5 years, per the Ontario ESA. T Industries provides an additional floating day after 2 years."),
 ("Statutory holidays", "BC: New Year's Day, Family Day, Good Friday, Victoria Day, Canada Day, BC Day, Labour Day, National Day for Truth and Reconciliation, Thanksgiving, Remembrance Day, Christmas Day. Ontario: as per the ESA, plus Boxing Day as a company holiday."),
 ("Warehouse safety", "Steel-toe boots and high-visibility vests are mandatory in the Delta BC and Toronto warehouses. Report all incidents within 24 hours (WorkSafeBC / WSIB)."),
 ("Expenses", "Mileage is reimbursed at the CRA prescribed rate. Submit expenses within 30 days.")],
 meta="Document owner: People & Culture | Revision 2026.1 | Internal - do not distribute")
# --- Business rules (markdown) ---
open(f"{SP}/Business Rules/net_sales_definition.md", "w").write("""# Business rule: Net sales
Owner: Finance (Controller) - approved 2026-02-01

**Net sales (CAD)** = gross sales - discounts - returns.

* **Gross sales** = shipped quantity x unit list price.
* **Discounts** = shipped quantity x (unit list price - unit net invoiced price).
* **Returns** = returned quantity x unit net invoiced price (credit memos).
* Excludes freight charges, GST/HST/PST and US sales tax.
* Cancelled orders are excluded entirely.
* All amounts are in CAD.

**Gross margin** = net sales - cost of goods sold, where COGS = (shipped quantity - returned quantity) x standard unit cost.
**Gross margin %** = gross margin / net sales.
**Average selling price (ASP)** = net sales / shipped units.
""")
open(f"{SP}/Business Rules/backorder_definition.md", "w").write("""# Business rule: Backorders
Owner: Operations (Director of Supply Chain) - approved 2026-02-01

* A **backorder** is any ordered quantity that could not be shipped from the assigned warehouse (Delta BC or Toronto) on the original ship date because of insufficient available stock.
* **Backorder rate** = backordered units / ordered units, on non-cancelled orders, measured by order date.
* Target: backorder rate below 5% per warehouse per month. Above 10% for a product category triggers a supply review.
* **Available stock** = on-hand quantity - allocated quantity. A SKU is **below reorder point** when available stock + on-order quantity is less than the reorder point.
""")
print(sum(len(f) for _, _, f in os.walk(SP)), "SharePoint files")

In [0]:
HA = f"{VOL}/hubspot_attachments"
if os.path.exists(HA): shutil.rmtree(HA)
os.makedirs(HA)
att = []
# 8 PDF quotes for exit-device pricing disputes (quote vs invoice)
disp = hubspot_tickets[hubspot_tickets.subject.str.startswith("Invoice discrepancy") & hubspot_tickets.sku_mentioned.fillna("").str.startswith("TH-EX")].head(8)
for t in disp.itertuples():
    p = pmap[t.sku_mentioned]; fn = f"Quote_Q-{t.ticket_id % 100000}_{t.sku_mentioned}.pdf"; qp = round(p["list_price_cad"] * 0.62, 2)
    pdf(f"{HA}/{fn}", f"T Industries Project Quote Q-{t.ticket_id % 100000}", [
        ("Customer", f"HubSpot company {t.company_id}. Reference order {t.erp_order_number or 'n/a'}. Quote valid 90 days."),
        ("Line items", [["SKU","Description","Qty","Quoted unit CAD","List CAD"],[t.sku_mentioned, p["name"][:40], "24", f"{qp:,.2f}", f"{p['list_price_cad']:,.2f}"]]),
        ("Notes", "Pricing matched to NorthBuild Supply web price per Regional Sales Manager approval. Freight and taxes extra.")],
        meta="Sales Quote | T Industries Ltd.")
    att.append(dict(ticket_id=int(t.ticket_id), deal_id=None, file_name=fn, content_type="application/pdf"))
# 8 ticket photos: finish defects and damaged cartons
for t in hubspot_tickets[hubspot_tickets.subject.str.startswith(("Finish peeling","Damaged carton"))].head(8).itertuples():
    sku = t.sku_mentioned or "TH-FC-1189"; p = dict(pmap.get(sku, pmap["TH-FC-1189"]), sku=sku)
    img = Image.new("RGB", (640,480), (120,110,100)); d = ImageDraw.Draw(img); draw_product(d, p["category"], FCOL[p["finish_code"]])
    for _ in range(40): x, y = int(rng.integers(150,450)), int(rng.integers(150,330)); d.ellipse([(x,y),(x+int(rng.integers(4,18)),y+int(rng.integers(4,12)))], fill=(200,190,170))
    d.text((20,440), f"Ticket {t.ticket_id} - site photo - {sku}", font=font(18), fill=(255,255,255))
    fn = f"ticket_{t.ticket_id}_photo.jpg"; img.save(f"{HA}/{fn}", "JPEG")
    att.append(dict(ticket_id=int(t.ticket_id), deal_id=None, file_name=fn, content_type="image/jpeg"))
# 4 screenshots of the customer web portal showing backorder status
for t in hubspot_tickets[hubspot_tickets.subject.str.startswith("Backorder ETA")].head(4).itertuples():
    img = Image.new("RGB", (900,420), (250,250,250)); d = ImageDraw.Draw(img)
    d.rectangle([(0,0),(900,50)], fill=(20,40,90)); d.text((20,12), "T Industries Customer Portal - Order status", font=font(22), fill=(255,255,255))
    d.text((30,80), f"Order {t.erp_order_number or 'SO-??????'}   SKU {t.sku_mentioned}", font=font(22), fill=(0,0,0))
    d.text((30,130), "Status: BACKORDERED - awaiting supplier receipt at Toronto DC", font=font(20), fill=(180,30,30))
    d.text((30,180), "Estimated availability: TBD", font=font(20), fill=(60,60,60))
    fn = f"ticket_{t.ticket_id}_portal_screenshot.png"; img.save(f"{HA}/{fn}", "PNG")
    att.append(dict(ticket_id=int(t.ticket_id), deal_id=None, file_name=fn, content_type="image/png"))
print(len(att), "HubSpot attachments")

In [0]:
from pyspark.sql import functions as F
# Mirrors the Lakeflow Connect SharePoint connector output (file metadata + raw bytes)
sp = (spark.read.format("binaryFile").option("recursiveFileLookup", "true").load(SP)
      .select(F.element_at(F.split("path", "/"), -1).alias("file_name"),
              F.regexp_replace("path", "^dbfs:", "").alias("path"),
              F.regexp_extract("path", r"/sharepoint/([^/]+)/", 1).alias("folder"),
              F.lit("T Industries Intranet / Shared Documents").alias("site"),
              F.col("modificationTime").alias("modified_time"), F.col("length").alias("size_bytes"), "content"))
sp.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{B}.bronze_sharepoint_files")
spark.sql(f"COMMENT ON TABLE {B}.bronze_sharepoint_files IS 'SharePoint document library files as landed by Lakeflow Connect (simulated): one row per file with raw bytes. Parsed by ai_parse_document in pipeline t_genie_ready.'")
for c, t in dict(file_name="File name", path="UC volume path of the file", folder="SharePoint folder (Product Spec Sheets, Installation SOPs, Customer Service SOPs, HR, Business Rules, Pricing)",
                 site="SharePoint site / library", modified_time="Last modified time", size_bytes="File size in bytes", content="Raw file bytes (PDF, DOCX or markdown)").items():
    spark.sql(f"ALTER TABLE {B}.bronze_sharepoint_files ALTER COLUMN {c} COMMENT {t!r}")

meta = spark.createDataFrame(pd.DataFrame(att).astype(object).where(pd.DataFrame(att).notna(), None),
                             "ticket_id long, deal_id long, file_name string, content_type string")
ha = (spark.read.format("binaryFile").load(HA)
      .select(F.element_at(F.split("path", "/"), -1).alias("file_name"), F.regexp_replace("path", "^dbfs:", "").alias("path"),
              F.col("modificationTime").alias("uploaded_at"), "content")
      .join(meta, "file_name")
      .select("ticket_id", "deal_id", "file_name", "content_type", "path", "uploaded_at", "content"))
ha.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{B}.bronze_hubspot_attachments")
spark.sql(f"COMMENT ON TABLE {B}.bronze_hubspot_attachments IS 'HubSpot ticket attachments (simulated Lakeflow Connect file output): PDF quotes, site photos and portal screenshots with raw bytes.'")
for c, t in dict(ticket_id="HubSpot ticket id", deal_id="HubSpot deal id (if attached to a deal)", file_name="Attachment file name", content_type="MIME type",
                 path="UC volume path", uploaded_at="Upload time", content="Raw file bytes").items():
    spark.sql(f"ALTER TABLE {B}.bronze_hubspot_attachments ALTER COLUMN {c} COMMENT {t!r}")

display(spark.sql(f"""SELECT table_name, comment FROM {C}.information_schema.tables WHERE table_schema = 't_hardware' AND table_name LIKE 'bronze_%' ORDER BY 1"""))
for t in ["products","customers","sales_orders","order_lines","inventory","competitor_prices","hubspot_companies","hubspot_deals","hubspot_tickets","hubspot_attachments","sharepoint_files"]:
    print(f"{t:22s}", spark.table(f"{B}.bronze_{t}").count())
sync_knowledge()   # refresh the raw-document volume with the regenerated SharePoint files